## Tugas Mandiri Pemrograman Paralel Pertemuan 5
**Nama: Shahida Shintya  
NIM: 2411511007**  

### Tugas 1 - Bank Account Race Condition

In [1]:
import threading
import time

class RekeningBank:
    def __init__(self, saldo_awal=1000000):
        self.saldo = saldo_awal
        self.pengunci = threading.Lock()

    # Fungsi tanpa lock (rawan race condition)
    def proses_tanpa_lock(self):
        for _ in range(10000):
            self.saldo -= 1
            self.saldo += 1

    # Fungsi dengan lock
    def proses_dengan_lock(self):
        for _ in range(10000):
            with self.pengunci:
                self.saldo -= 1
                self.saldo += 1

# Simulasi Tanpa Lock
rekening_A = RekeningBank(1000000)
daftar_thread_1 = []
for _ in range(10):
    t = threading.Thread(target=rekening_A.proses_tanpa_lock)
    daftar_thread_1.append(t)
    t.start()
for t in daftar_thread_1:
    t.join()
print(f"Hasil saldo akhir tanpa lock: {rekening_A.saldo}")

# Simulasi Dengan Lock
rekening_B = RekeningBank(1000000)
daftar_thread_2 = []
for _ in range(10):
    t = threading.Thread(target=rekening_B.proses_dengan_lock)
    daftar_thread_2.append(t)
    t.start()
for t in daftar_thread_2:
    t.join()
print(f"Hasil saldo akhir dengan lock: {rekening_B.saldo}")

Hasil saldo akhir tanpa lock: 1000000
Hasil saldo akhir dengan lock: 1000000


**Penjelasan**                      
Di nomor ini kita bikin simulasi saldo awal satu juta dengan sepuluh thread yang masing-masing narik dan setor satu rupiah sebanyak sepuluh ribu kali. Kalau gak pake lock, hasilnya bakal berantakan alias gak pas satu juta lagi karena kena race condition (saling tumpang tindih saat ngubah variabel saldo). Makanya kita butuh threading.Lock() biar proses ubah saldonya antre satu-satu secara eksklusif, bikin saldo akhirnya aman konsisten di angka satu juta.

### Tugas 2 - Poducer-Consumer dengan Banyak Consumer

In [2]:
import threading
import queue
import time
import random

antrean = queue.Queue()

def produsen(q, jumlah_data, total_konsumen):
    for i in range(1, jumlah_data + 1):
        item = f"Paket-{i}"
        time.sleep(random.uniform(0.05, 0.15))
        q.put(item)
        print(f"[Produsen] Mengirim: {item}")
    
    # Kirim sinyal None sebanyak jumlah konsumen agar semua thread berhenti
    for _ in range(total_konsumen):
        q.put(None)
    print("[Produsen] Selesai.")

def konsumen(nama_konsumen, q):
    while True:
        item = q.get()
        if item is None:
            q.task_done()
            break
        time.sleep(random.uniform(0.1, 0.2))
        print(f"   [{nama_konsumen}] Memproses: {item}")
        q.task_done()
    print(f"[{nama_konsumen}] Berhenti.")

# Inisialisasi thread
jml_konsumen = 3
t_produsen = threading.Thread(target=produsen, args=(antrean, 9, jml_konsumen))

t_list_konsumen = []
for i in range(1, jml_konsumen + 1):
    t = threading.Thread(target=konsumen, args=(f"Konsumen-{i}", antrean))
    t_list_konsumen.append(t)

# Eksekusi
t_produsen.start()
for t in t_list_konsumen:
    t.start()

t_produsen.join()
antrean.join()
for t in t_list_konsumen:
    t.join()

print("Semua proses produsen dan konsumen beres.")

[Produsen] Mengirim: Paket-1
[Produsen] Mengirim: Paket-2
   [Konsumen-1] Memproses: Paket-1
   [Konsumen-2] Memproses: Paket-2
[Produsen] Mengirim: Paket-3
[Produsen] Mengirim: Paket-4
   [Konsumen-3] Memproses: Paket-3
[Produsen] Mengirim: Paket-5
   [Konsumen-1] Memproses: Paket-4
   [Konsumen-2] Memproses: Paket-5
[Produsen] Mengirim: Paket-6
[Produsen] Mengirim: Paket-7
   [Konsumen-3] Memproses: Paket-6
[Produsen] Mengirim: Paket-8   [Konsumen-1] Memproses: Paket-7

[Produsen] Mengirim: Paket-9
[Produsen] Selesai.
[Konsumen-2] Berhenti.
   [Konsumen-1] Memproses: Paket-8
[Konsumen-1] Berhenti.
   [Konsumen-3] Memproses: Paket-9
[Konsumen-3] Berhenti.
Semua proses produsen dan konsumen beres.


**Penjelasan**     
Disini kita nerapin pola Producer-Consumer tapi dengan satu produsen dan tiga konsumen sekaligus. Biar tiga konsumen ini tau kapan harus berhenti kerja tanpa ada yang ngegantung, si produsen bakal ngirim sinyal None sebanyak tiga kali di akhir antrean. Begitu konsumen narik data dan dapet nilai None, mereka langsung mutusin loop dan berhenti dengan bersih.

### Tugas 3 - Eksperimen ThreadPoolExecutor

In [3]:
import concurrent.futures
import time

def tugas_unduh(id_tugas):
    time.sleep(0.08) # simulasi delay
    return f"Tugas {id_tugas} sukses"

jumlah_total_file = 50
pengujian_worker = [2, 4, 8, 16]

for mw in pengujian_worker:
    waktu_mulai = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=mw) as eksikutor:
        future_list = [eksikutor.submit(tugas_unduh, n) for n in range(1, jumlah_total_file + 1)]
        concurrent.futures.wait(future_list)
        
    waktu_selesai = time.time()
    total_durasi = waktu_selesai - waktu_mulai
    print(f"Kapasitas max_workers = {mw:2d} | Durasi Total: {total_durasi:.4f} detik")

Kapasitas max_workers =  2 | Durasi Total: 2.0176 detik
Kapasitas max_workers =  4 | Durasi Total: 1.0488 detik
Kapasitas max_workers =  8 | Durasi Total: 0.5638 detik
Kapasitas max_workers = 16 | Durasi Total: 0.3240 detik


**Penjelasan**            
Kode ini ngetes performa ThreadPoolExecutor buat ngurusin 50 tugas pake variasi max_workers 2, 4, 8, dan 16. Penambahan workers gak bakal selamanya bikin makin cepat karena ada titik jenuh (bottleneck). Kalau jumlah thread-nya kelebihan, CPU malah bakal kebanyakan ngabisin waktu buat nata antrean (context switching) ketimbang ngerjain tugas intinya.

### Tugas 4 - Refleksi Singkat
**Contoh kasus di mana threading Python membantu:** Saat kita bikin program pengunduh massal (bulk downloader) atau web scraper yang narik banyak halaman web sekaligus lewat internet. Threading ngebantu banget karena program gak perlu nunggu satu file kelar dulu buat mulai narik file berikutnya, jadi proses I/O-nya jauh lebih ngebut.

**Contoh kasus di mana threading tidak membantu dan sebaiknya menunggu multiprocessing:** Waktu kita mau ngolah video/gambar berat atau ngitung rumus matematika kompleks secara paralel (CPU-bound). Di Python, hal ini gak bakal dipercepat sama threading gara-gara ada aturan GIL (Global Interpreter Lock) yang bikin eksekusi kode mutlak cuma jalan di satu inti CPU doang, sehingga tugas-tugas berat kayak gini wajib pakai multiprocessing.